# Training YOLOv8n - Deteksi Makanan (UEC FOOD 256)

Notebook ini melatih model **YOLOv8n** untuk mendeteksi 256 jenis makanan
dari dataset UEC FOOD 256 (format YOLO).

## Sebelum klik "Run All" - WAJIB:
1. **Attach dataset**: panel kanan -> *Input* -> *Add Input* -> cari & tambahkan
   dataset `yolo-uec-food-256-v2`.
2. **Aktifkan GPU**: panel kanan -> *Settings* -> *Accelerator* -> pilih
   **GPU T4 x2** (atau P100).
3. Klik **Run All**. Estimasi waktu sekitar 2-3 jam.

Hasil akhir: bobot model `best.pt` tersimpan di tab *Output* dan bisa diunduh.

In [ ]:
# 1. Install Ultralytics (YOLOv8)
!pip install -q -U ultralytics

In [ ]:
# 2. Setup & cek GPU
import torch, ultralytics
from ultralytics import YOLO

print("Ultralytics :", ultralytics.__version__)
print("PyTorch     :", torch.__version__)

if not torch.cuda.is_available():
    raise SystemError(
        "GPU TIDAK AKTIF. Buka panel kanan -> Settings -> Accelerator -> pilih GPU, "
        "lalu jalankan ulang notebook (Run All)."
    )
print("GPU         :", torch.cuda.get_device_name(0))

In [ ]:
# 3. Auto-detect folder dataset di dalam /kaggle/input
import os, glob

DEFAULT_ROOT = "/kaggle/input/yolo-uec-food-256-v2"

def find_dataset_root():
    """Cari folder yang berisi images/train DAN labels/train."""
    for base in [DEFAULT_ROOT, "/kaggle/input"]:
        if not os.path.isdir(base):
            continue
        for train_dir in glob.glob(os.path.join(base, "**", "images", "train"),
                                   recursive=True):
            root = os.path.dirname(os.path.dirname(train_dir))
            if os.path.isdir(os.path.join(root, "labels", "train")):
                return root
    return None

DATA_ROOT = find_dataset_root()
if DATA_ROOT is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan di /kaggle/input. Pastikan dataset "
        "'yolo-uec-food-256-v2' sudah di-attach lewat panel Input di kanan."
    )

print("Dataset root:", DATA_ROOT)
for split in ["train", "val", "test"]:
    d = os.path.join(DATA_ROOT, "images", split)
    print(f"  images/{split:5s}: {'OK' if os.path.isdir(d) else 'MISSING'}  ->  {d}")

In [ ]:
# 4. Buat data.yaml khusus Kaggle (path absolut)
import yaml

orig_yaml = os.path.join(DATA_ROOT, "data.yaml")
if not os.path.isfile(orig_yaml):
    raise FileNotFoundError(f"data.yaml tidak ditemukan di {DATA_ROOT}")

with open(orig_yaml) as f:
    meta = yaml.safe_load(f)
names = meta["names"]
nc = meta.get("nc", len(names))
print(f"data.yaml asli ditemukan: nc={nc}, jumlah nama kelas={len(names)}")

data_cfg = {
    "path":  DATA_ROOT,
    "train": os.path.join(DATA_ROOT, "images", "train"),
    "val":   os.path.join(DATA_ROOT, "images", "val"),
    "test":  os.path.join(DATA_ROOT, "images", "test"),
    "nc":    nc,
    "names": names,
}

DATA_YAML = "/kaggle/working/data_kaggle.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)
print("data.yaml Kaggle ditulis ke:", DATA_YAML)

In [ ]:
# 5. Validasi dataset (pre-flight) sebelum training
import random

def count_files(d):
    return len(glob.glob(os.path.join(d, "*"))) if os.path.isdir(d) else 0

print("Jumlah file per split:")
all_ok = True
for split in ["train", "val", "test"]:
    n_img = count_files(os.path.join(DATA_ROOT, "images", split))
    n_lbl = count_files(os.path.join(DATA_ROOT, "labels", split))
    ok = n_img == n_lbl and n_img > 0
    all_ok &= ok
    print(f"  {split:5s}: {n_img:6d} gambar | {n_lbl:6d} label  [{'OK' if ok else 'PERIKSA!'}]")

# Cek sampel isi label
label_files = glob.glob(os.path.join(DATA_ROOT, "labels", "train", "*.txt"))
sample = random.sample(label_files, min(300, len(label_files)))
bad = 0
for lf in sample:
    with open(lf) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = line.split()
            if len(p) != 5:
                bad += 1
                continue
            cid = int(float(p[0]))
            xywh = [float(v) for v in p[1:]]
            if not (0 <= cid < nc) or any(not (0.0 <= v <= 1.0) for v in xywh):
                bad += 1

print(f"\nCek {len(sample)} file label sampel: {bad} baris bermasalah")
if bad or not all_ok:
    print("PERINGATAN: ada yang perlu diperiksa pada dataset.")
else:
    print("Dataset valid - siap untuk training.")

In [ ]:
# 6. Training YOLOv8n (transfer learning dari bobot COCO)
model = YOLO("yolov8n.pt")

model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=64,          # turunkan ke 32 jika muncul "CUDA out of memory"
    device=0,
    patience=20,       # early stopping bila tidak membaik 20 epoch
    cache=False,
    workers=4,
    seed=42,
    plots=True,
    project="/kaggle/working/runs",
    name="food_yolov8n",
    exist_ok=True,
)

SAVE_DIR = str(model.trainer.save_dir)
BEST_PT = os.path.join(SAVE_DIR, "weights", "best.pt")
print("Training selesai. Folder hasil:", SAVE_DIR)

In [ ]:
# 7. Evaluasi di TEST set
best_model = YOLO(BEST_PT)

metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    device=0,
    project="/kaggle/working/runs",
    name="food_yolov8n_test",
    exist_ok=True,
)
print(f"mAP50    (test): {metrics.box.map50:.4f}")
print(f"mAP50-95 (test): {metrics.box.map:.4f}")

# Tampilkan grafik hasil training
from IPython.display import Image, display
for img_name in ["results.png", "confusion_matrix_normalized.png"]:
    p = os.path.join(SAVE_DIR, img_name)
    if os.path.isfile(p):
        print("\n" + img_name)
        display(Image(filename=p))

In [ ]:
# 8. Visualisasi contoh prediksi pada gambar test
import matplotlib.pyplot as plt

test_imgs = glob.glob(os.path.join(DATA_ROOT, "images", "test", "*"))
sample_imgs = random.sample(test_imgs, min(6, len(test_imgs)))

preds = best_model.predict(sample_imgs, imgsz=640, conf=0.25,
                           device=0, verbose=False)

plt.figure(figsize=(16, 9))
for i, r in enumerate(preds):
    plt.subplot(2, 3, i + 1)
    plt.imshow(r.plot()[..., ::-1])   # konversi BGR -> RGB
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 9. Simpan bobot final agar mudah diunduh
import shutil

DST = "/kaggle/working/best.pt"
shutil.copy(BEST_PT, DST)
print("Bobot final disalin ke:", DST)
print("Unduh file 'best.pt' dari tab 'Output' notebook setelah selesai.")